In [ ]:
# =============================================================================
# Zelle 01 – Setup Phase 2: Pfade, Stil, Eingaben aus der EDA laden und prüfen
# =============================================================================
# Zweck: Grundlage für Gold-Set (Pilot, Umfang) und Split (gruppiert nach Ordner,
#   Entscheidung 12.11.2026). Notebook ist eigenständig lauffähig ("Run All"):
#   Eingaben sind die gespeicherten Zwischenstände aus Notebook 02, nicht dessen Kernel.
# Prüfung: Zeilenzahlen gegen die Befunde aus Notebook 02 (Assertions).
# Hinweis: tabelle_speichern() ist hier ein zweites Mal definiert (wie Notebook 02);
#   Umzug nach src/ in Phase 3.
# =============================================================================

import ast
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

from playlist_agent.viz_config import ARTEFAKT_MUSTER, FARBEN, figur_speichern, stil_aktivieren

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
assert PROJECT_ROOT.name == "_MyMSIT_AI-Agent_repair-playlist", PROJECT_ROOT

RAW, INTERIM = PROJECT_ROOT / "data/raw", PROJECT_ROOT / "data/interim"
GOLD = PROJECT_ROOT / "data/gold"
FIG, TAB, ERR = (PROJECT_ROOT / "reports" / p for p in ("figures", "tables", "errors"))
SEED = 42
cfg = yaml.safe_load((PROJECT_ROOT / "config/config.yaml").read_text(encoding="utf-8"))
stil_aktivieren()


def tabelle_speichern(df: pd.DataFrame, name: str, ordner: Path = TAB) -> Path:
    if not ARTEFAKT_MUSTER.match(name):
        raise ValueError(f"Name '{name}' entspricht nicht <NB>_<Zelle>_<slug>.")
    pfad = ordner / f"{name}.csv"
    df.to_csv(pfad, index=False, encoding="utf-8")
    return pfad


def nfc(s: str) -> str:
    return unicodedata.normalize("NFC", s)


# --- Eingaben aus Notebook 02 (privat) -------------------------------------------
df_crate_fehl = pd.read_csv(ERR / "02_06_fehltreffer_kategorien.csv")
df_crate_fehl["kandidaten"] = df_crate_fehl["kandidaten"].map(ast.literal_eval)   # Liste war als Text gespeichert
df_pl_kanon = pd.read_csv(INTERIM / "02_17_playlist_eintraege_kanonisch.csv")
df_pl_a1 = pd.read_csv(ERR / "02_19_playlist_a1_praefix.csv")
df_pl_fehl = pd.read_csv(ERR / "02_18_playlist_fehltreffer.csv")
df_tags = pd.read_csv(INTERIM / "02_12_audio_tags.csv")
df_db = pd.read_csv(INTERIM / "02_08_database_v2.csv", low_memory=False)
df_null = pd.read_csv(ERR / "02_25_null_byte_ordner_liste.csv")
df_nutzer = pd.read_csv(GOLD / "02_15_nutzer_labels.csv")

# --- Prüfung gegen Befunde Notebook 02 ---------------------------------------------
ERWARTET = {"df_crate_fehl": (df_crate_fehl, 404), "df_pl_kanon": (df_pl_kanon, 10289),
            "df_pl_a1": (df_pl_a1, 3000), "df_pl_fehl": (df_pl_fehl, 3942),
            "df_tags": (df_tags, 5236), "df_db": (df_db, 5478),
            "df_null": (df_null, 23), "df_nutzer": (df_nutzer, 10)}
for name, (df, n) in ERWARTET.items():
    status = "OK " if len(df) == n else "ABWEICHUNG"
    print(f"{status} {name:<14} {len(df):>6} Zeilen (erwartet {n})")
assert all(len(df) == n for df, n in ERWARTET.values()), "Eingaben weichen von Notebook 02 ab"

In [ ]:
# =============================================================================
# Zelle 02 – Gold-Pilot: geschichtete Stichprobe (20 Fälle, Kandidaten-Prüfung)
# =============================================================================
# Entscheidung 12.11.2026 (Option a): K3 7 | Q2 7 | Q3 6 – alle Fälle haben genau
#   einen Kandidaten (Aufgabentyp "Kandidat prüfen"). Ziel: Zeit je Label messen.
# Obergrenze: max. 2 Fälle je Ordner (Gruppe = Ordner des kanonischen Pfads),
#   damit kein Ordner den Pilot dominiert (Konzentration bis 49 %, NB 02 Zelle 23).
# Ziehung: Mischen mit SEED, je Ordner die ersten 2, dann die ersten n der Schicht.
# Datenschutz: Auswahl mit Pfaden privat (data/gold), public nur Zählwerte.
# =============================================================================

import posixpath

KAT_A = "a Name 1x im Index, anderer Ordner"
PLAN = {"K3": 7, "Q2": 7, "Q3": 6}
MAX_JE_ORDNER = 2

# --- Grundgesamtheiten (eindeutig je kanonischem Pfad) -------------------------------
k3 = df_crate_fehl[df_crate_fehl["kategorie"] == KAT_A].assign(
    schicht="K3", kanonisch=lambda d: d["pfad"].map(nfc),
    eintrag=lambda d: d["pfad"], kandidat=lambda d: d["kandidaten"].map(lambda k: k[0]))
q = df_pl_a1.assign(kanonisch=lambda d: d["kanonisch"].map(nfc))
q2 = q[q["gleiche_endteile"] >= 2].assign(schicht="Q2")
q3 = q[q["gleiche_endteile"] == 1].assign(schicht="Q3")
spalten = ["schicht", "kanonisch", "eintrag", "kandidat"]
basis = pd.concat([k3[spalten], q2[spalten], q3[spalten]]).drop_duplicates(["schicht", "kanonisch"])
basis["ordner"] = basis["kanonisch"].map(posixpath.dirname)

# --- Ziehung mit Obergrenze je Ordner ---------------------------------------------------
teile = []
for schicht, n in PLAN.items():
    g = basis[basis["schicht"] == schicht].sample(frac=1, random_state=SEED)
    g = g.groupby("ordner", sort=False).head(MAX_JE_ORDNER).head(n)
    teile.append(g)
df_pilot = pd.concat(teile).reset_index(drop=True)
df_pilot.insert(0, "nr", [f"P{i:02d}" for i in range(1, len(df_pilot) + 1)])
df_pilot.to_csv(GOLD / "03_02_pilot_auswahl.csv", index=False, encoding="utf-8")   # privat

df_pilot_zaehlung = (df_pilot.groupby("schicht", as_index=False)
                     .agg(faelle=("nr", "size"), ordner=("ordner", "nunique"),
                          max_je_ordner=("ordner", lambda s: s.value_counts().max())))
df_pilot_zaehlung["grundgesamtheit"] = df_pilot_zaehlung["schicht"].map(
    basis["schicht"].value_counts())
print(df_pilot_zaehlung.to_string(index=False))
_ = tabelle_speichern(df_pilot_zaehlung, "03_02_pilot_zaehlung")

In [ ]:
# =============================================================================
# Zelle 03 – Gold-Pilot: Labeln mit Zeitmessung je Fall (Eingabe r / f / u)
# =============================================================================
# Protokoll (Entscheidung 12.11.2026): je Fall Kandidat im Explorer/foobar prüfen,
#   Urteil r = richtige Datei | f = falsche Datei | u = unklar (+ optional Kommentar).
# Zeit je Fall = Anzeige bis Eingabe (perf_counter) -> Median, Spanne, Lerneffekt.
# Robust: jedes Label sofort gespeichert; bereits gelabelte Fälle werden übersprungen
#   (Fortsetzen nach Unterbrechung). Keine Datei wird überschrieben, nur ergänzt.
# Laufwerk aus config.yaml über den Hostnamen (Asus D:, Medion E:).
# Datenschutz: Labels mit Pfaden privat (data/gold).
# =============================================================================

import socket
import time

LABEL_DATEI = GOLD / "03_03_pilot_labels.csv"
LW = next(l["laufwerk_beobachtet"] for l in cfg["laptops"] if l["hostname"] == socket.gethostname())
tag_dauer = dict(zip(df_tags["pfad_relativ"].map(nfc), df_tags["duration"]))
db_idx = df_db.dropna(subset=["schluessel"]).drop_duplicates("schluessel").set_index("schluessel")

erledigt = set(pd.read_csv(LABEL_DATEI)["nr"]) if LABEL_DATEI.exists() else set()
offen = df_pilot[~df_pilot["nr"].isin(erledigt)]
print(f"Fälle gesamt {len(df_pilot)} | erledigt {len(erledigt)} | offen {len(offen)}\n")

for _, z in offen.iterrows():
    t0 = time.perf_counter()
    d = tag_dauer.get(nfc(z["kandidat"]))
    print("=" * 80)
    print(f"{z['nr']}  Schicht {z['schicht']}")
    print(f"  Playlist-Eintrag : {z['eintrag']}")
    print(f"  Kandidat         : {LW}\\{z['kandidat'].replace('/', chr(92))}")
    print(f"  Datei-Dauer      : {d:.2f} s" if pd.notna(d) else "  Datei-Dauer      : –")
    k = z["kanonisch"]
    if k in db_idx.index:
        print(f"  Serato           : Titel={db_idx.at[k, 'tsng']!r} | Dauer={db_idx.at[k, 'tlen']!r}")
    antwort = ""
    while antwort[:1].lower() not in ("r", "f", "u"):
        antwort = input(f"{z['nr']} Urteil r/f/u (+ Kommentar): ").strip()
    sekunden = time.perf_counter() - t0
    zeile = pd.DataFrame([{"nr": z["nr"], "schicht": z["schicht"], "kanonisch": k,
                           "kandidat": z["kandidat"], "label": antwort[0].lower(),
                           "kommentar": antwort[1:].strip(), "sekunden": round(sekunden, 1)}])
    zeile.to_csv(LABEL_DATEI, mode="a", header=not LABEL_DATEI.exists(), index=False, encoding="utf-8")
    print(f"  -> gespeichert: {antwort[0].lower()} ({sekunden:.0f} s)\n")

print("Pilot vollständig gelabelt." if len(pd.read_csv(LABEL_DATEI)) == len(df_pilot) else "Unterbrochen – Zelle erneut starten.")

In [ ]:
# =============================================================================
# Zelle 04 – Gold-Pilot: Auswertung (Labels, Zeit je Fall, Hochrechnung)
# =============================================================================
# 1 Labels je Schicht (r/f/u) – nur Warnsignal, keine Precision-Aussage
#   (n = 6–7 je Schicht; Rule of Three: 0 Fehler in 7 -> nur < 43 %).
# 2 Zeit je Fall: Median, Mittel, Quartile, Spanne; Bootstrap-95-%-KI des Medians.
# 3 Lerneffekt: Zeit erste 10 vs. letzte 10 Fälle (Reihenfolge der Bearbeitung).
# 4 Hochrechnung Aufwand für Schichten vom Typ "Kandidat prüfen"
#   (K1, K2, K3, Q1, Q2, Q3, Q4a; Umfänge aus NB 02 Zelle 23) bei
#   Zielgenauigkeit < 5 / 3 / 1 % – mit Median und KI-Grenzen.
#   Suchfälle (K4, Q4b, Q5) ausgenommen: eigener Pilot nötig.
# =============================================================================

df_labels = pd.read_csv(LABEL_DATEI)
assert len(df_labels) == 20, f"{len(df_labels)} von 20 gelabelt"

# --- 1. Labels ---------------------------------------------------------------------
df_pilot_labels = pd.crosstab(df_labels["schicht"], df_labels["label"], margins=True, margins_name="Summe")
print("=== 1: Labels je Schicht (r richtig | f falsch | u unklar) ===")
print(df_pilot_labels.to_string())
_ = tabelle_speichern(df_pilot_labels.reset_index(), "03_04_pilot_labels")

# --- 2. Zeit je Fall ---------------------------------------------------------------
s = df_labels["sekunden"].to_numpy()
rng = np.random.default_rng(SEED)
boot_median = np.array([np.median(rng.choice(s, len(s), replace=True)) for _ in range(10_000)])
ki_u, ki_o = np.percentile(boot_median, [2.5, 97.5])
q1, med, q3 = np.percentile(s, [25, 50, 75])
print("\n=== 2: Zeit je Fall (s) ===")
print(f"Median {med:.0f} (95-%-KI {ki_u:.0f}–{ki_o:.0f}) | Mittel {s.mean():.0f} | "
      f"Quartile {q1:.0f}–{q3:.0f} | Spanne {s.min():.0f}–{s.max():.0f}")
print(df_labels.groupby("schicht")["sekunden"].median().round(0).rename("median_s").to_string())

# --- 3. Lerneffekt ----------------------------------------------------------------
print(f"\n=== 3: Lerneffekt === Median erste 10: {np.median(s[:10]):.0f} s | "
      f"letzte 10: {np.median(s[10:]):.0f} s")

# --- 4. Hochrechnung ---------------------------------------------------------------
df_sch = pd.read_csv(TAB / "02_23_goldset_schichten.csv")
typ_kandidat = df_sch["schicht"].str.match(r"^(K1|K2|K3|Q1|Q2|Q3|Q4a)\b")
df_aufwand = pd.DataFrame([
    {"ziel": ziel, "labels": int(df_sch.loc[typ_kandidat, spalte].sum()),
     "stunden_median": round(df_sch.loc[typ_kandidat, spalte].sum() * med / 3600, 1),
     "stunden_ki_unten": round(df_sch.loc[typ_kandidat, spalte].sum() * ki_u / 3600, 1),
     "stunden_ki_oben": round(df_sch.loc[typ_kandidat, spalte].sum() * ki_o / 3600, 1)}
    for ziel, spalte in [("< 5 %", "n_5pct"), ("< 3 %", "n_3pct"), ("< 1 %", "n_1pct")]])
print("\n=== 4: Aufwand Schichten 'Kandidat prüfen' (K1 K2 K3 Q1 Q2 Q3 Q4a) ===")
print(df_aufwand.to_string(index=False))
_ = tabelle_speichern(df_aufwand, "03_04_goldset_aufwand")

# --- Grafik: Zeit je Fall in Bearbeitungsreihenfolge -------------------------------
import matplotlib.pyplot as plt

farbe = {"K3": FARBEN["gold"], "Q2": FARBEN["blau"], "Q3": FARBEN["gruen"]}
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(df_labels["nr"], df_labels["sekunden"], color=df_labels["schicht"].map(farbe))
ax.axhline(med, color=FARBEN["text"], linestyle="--", label=f"Median {med:.0f} s")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=f, label=s) for s, f in farbe.items()]
          + [plt.Line2D([], [], color=FARBEN["text"], linestyle="--", label=f"Median {med:.0f} s")])
ax.set_ylabel("Sekunden je Label")
ax.set_title("Gold-Pilot: Zeit je Label")
ax.legend()
figur_speichern(fig, "03_04_pilot_zeit_je_fall", FIG)
plt.show()

### Befund und Entscheidung Zellen 02–04 – Gold-Pilot (Entscheidung 12.11.2026)

**Aufbau:** 20 Fälle vom Typ „Kandidat prüfen" (K3 7, Q2 7, Q3 6), max. 2 je Ordner, Label r/f/u mit Zeitmessung je Fall.

**Ergebnis:** 20/20 richtig (Rule of Three: Fehlerrate < 15 % gesamt, je Schicht nur < 43 % → Warnsignal, keine Precision-Aussage).
Zeit je Label: Median 21 s (Bootstrap-95-%-KI 18–33 s), Quartile 16–35 s, Spanne 14–221 s; erste 10 Fälle im Median 36 s, letzte 10 Fälle 18 s.

**Grenze des Versuchsplans:** Bearbeitung nach Schicht sortiert (K3 → Q2 → Q3) → Lerneffekt und Schichtunterschied (K3 52 s vs. Q3 16 s) nicht trennbar. Künftig zufällige Reihenfolge.

**Entscheidung:** Kein umfangreiches Labeln in Phase 2. Precision der Klasse „sicher" (P5) wird gemessen, wo sie entsteht: Stichprobe aus der von der Baseline (Phase 3) als „sicher" klassifizierten Menge, zufällige Reihenfolge, Blöcke zu 50 Labels (ca. 20 min), Abbruch bei erstem Fehler. Ziel 300 fehlerfreie Labels (< 1 %), Aufwand ca. 1,75 h.
**Begründung:** misst genau die geforderte Größe; exakte Pfade (K1, K2, Q1) sind per Konstruktion richtig und brauchen keine Labels; Aufwand ca. 1,75 h statt ca. 5 h.

In [ ]:
# =============================================================================
# Zelle 05 – Split: Zuordnung zu 5 Folds, gruppiert nach Ordner (Hash)
# =============================================================================
# Entscheidung 12.11.2026: gruppierter Split nach Ordner des kanonischen Pfads
#   (Leakage-Schutz; Konzentration bis 49 % je Ordner, NB 02 Zelle 23).
# Rechenvorschrift: fold = int(SHA-1(Ordner), 16) mod 5 -> stabil, unabhängig von
#   Reihenfolge und SEED; ein Ordner liegt immer in genau einem Fold.
#   Fold 0 = Testmenge (Erwartung ca. 20 %), Folds 1–4 = Training/Validierung.
# Grundgesamtheit: alle Fehltreffer (nur für lernende Komponenten relevant):
#   K3 Crates a | K4 Crates d/e | Q2/Q3 übrige A1 | Q4 übrige AM | Q5 übrige N.
# Prüfung: Anteil Fold 0 je Schicht – bei starker Ordner-Konzentration kann er
#   deutlich von 20 % abweichen (Messung, keine Annahme).
# Datenschutz: Zuordnung mit Pfaden privat (data/processed), public nur Zählwerte.
# =============================================================================

import hashlib

N_FOLDS = 5


def fold_von(ordner: str) -> int:
    return int(hashlib.sha1(ordner.encode("utf-8")).hexdigest(), 16) % N_FOLDS


teile = [
    df_crate_fehl.assign(schicht=np.where(df_crate_fehl["kategorie"] == KAT_A, "K3", "K4"),
                         kanonisch=df_crate_fehl["pfad"].map(nfc))[["schicht", "kanonisch"]],
    df_pl_a1.assign(schicht=np.where(df_pl_a1["gleiche_endteile"] >= 2, "Q2", "Q3"),
                    kanonisch=df_pl_a1["kanonisch"].map(nfc))[["schicht", "kanonisch"]],
    df_pl_fehl[df_pl_fehl["kategorie"] == "AM Name mehrfach anderswo"]
        .assign(schicht="Q4", kanonisch=lambda d: d["kanonisch"].map(nfc))[["schicht", "kanonisch"]],
    df_pl_fehl[df_pl_fehl["kategorie"] == "N Name nicht im Index"]
        .assign(schicht="Q5", kanonisch=lambda d: d["kanonisch"].map(nfc))[["schicht", "kanonisch"]],
]
df_split = pd.concat(teile).drop_duplicates(["schicht", "kanonisch"]).reset_index(drop=True)
df_split["ordner"] = df_split["kanonisch"].map(posixpath.dirname)
df_split["fold"] = df_split["ordner"].map(fold_von)
df_split.to_csv(PROJECT_ROOT / "data/processed/03_05_split_folds.csv", index=False, encoding="utf-8")  # privat

# --- Prüfung: Leakage und Verteilung -----------------------------------------------
assert df_split.groupby("ordner")["fold"].nunique().max() == 1, "Ordner in mehreren Folds"
df_split_zaehlung = (df_split.assign(test=df_split["fold"] == 0)
                     .groupby("schicht", as_index=False)
                     .agg(faelle=("kanonisch", "size"), ordner=("ordner", "nunique"),
                          test_faelle=("test", "sum"),
                          test_ordner=("ordner", lambda s: s[df_split.loc[s.index, "fold"] == 0].nunique())))
df_split_zaehlung["test_anteil"] = (df_split_zaehlung["test_faelle"] / df_split_zaehlung["faelle"]).round(3)
print(df_split_zaehlung.to_string(index=False))
print(f"\nGesamt: {len(df_split)} Fälle | Test (Fold 0): {(df_split['fold'] == 0).sum()} "
      f"({(df_split['fold'] == 0).mean():.1%})")
_ = tabelle_speichern(df_split_zaehlung, "03_05_split_zaehlung")

### Befund und Entscheidung Zelle 05 – Split nach Ordner (Entscheidung 12.11.2026)

**Rechenvorschrift:** fold = SHA-1(Ordner) mod 5; Fold 0 = Test. 1.887 eindeutige Fehltreffer; kein Ordner in mehreren Folds (geprüft).

**Ergebnis Testanteil:** Q2 23,2 % | Q3 23,1 % | Q5 21,9 % (ausgewogen) – K3 44,6 % (11 Ordner) | K4 11,4 % (15 Ordner) | Q4 2,9 % (10 Ordner, 1 Testfall).
Bei wenigen Ordnern entscheidet ein einzelner großer Ordner über die Testmenge → fester Split für K3/K4/Q4 nicht verlässlich.

**Entscheidung:** Hash-Split bleibt für Q2/Q3/Q5. Für Schichten mit wenigen Ordnern in Phase 4 Leave-One-Group-Out-Kreuzvalidierung (jeder Ordner einmal Testmenge). Relevanz begrenzt: lernende Verfahren betreffen nur den Rest ohne eindeutigen Namenskandidaten (EDA: ca. 394 + 33 Fälle).

In [ ]:
# =============================================================================
# Zelle 06 – Synthetische Testfälle T1–T8 (bekannte Antwort, ohne Handlabels)
# =============================================================================
# Quelle: alle Audiodateien des Index (Pfad = Datei, Antwort bekannt).
# Typen (Entscheidung 12.11.2026, aus EDA-Befunden):
#   T1 NFD | T2 fremdes Präfix/Laufwerk | T3 Ordnername + " old" | T4 anderer
#   bestehender Ordner | T5 Kleinschreibung | T6 Mojibake im Dateinamen
#   -> Erwartung: Originaldatei
#   T7 Zahl im Dateinamen geändert | T8 Jahreszahl in Pfad UND Dateiname geändert
#   -> Erwartung: "fehlt" (harte Negativfälle; Zuordnung wäre Fehler gegen P5)
# Bedingungen: T1/T6 nur bei Sonderzeichen; T3/T4 nur bei eindeutigem Dateinamen;
#   T7/T8 nur, wenn der neue Name nirgends im Index existiert.
# Umfang: 300 je Typ, max. 2 je Ordner (SEED); weniger, falls nicht genug geeignete Fälle.
# Datenschutz: privat (data/interim); Präfixe in T2 generisch.
# =============================================================================

import re

N_JE_TYP = 300
pool = pd.DataFrame({"pfad": df_tags["pfad_relativ"].map(nfc)})
pool["ordner"] = pool["pfad"].map(posixpath.dirname)
pool["name"] = pool["pfad"].map(posixpath.basename)
name_anz = pool["name"].str.lower().value_counts()
pool["name_eindeutig"] = pool["name"].str.lower().map(name_anz) == 1
alle_namen = set(pool["name"].str.lower())
alle_pfade_lower = set(pool["pfad"].str.lower())
alle_ordner = set(pool["ordner"])
ordner_liste = sorted(alle_ordner)
rng_syn = np.random.default_rng(SEED)
PRAEFIXE = ["Q:\\Music\\", "I:\\Audio\\", "C:\\Users\\dj\\Music\\"]


def t1(p):
    q = unicodedata.normalize("NFD", p)
    return q if q != p else None


def t2(p):
    rest = p.split("/", 1)[1]                                  # ohne "Musik/"
    return PRAEFIXE[rng_syn.integers(len(PRAEFIXE))] + rest.replace("/", "\\")


def t3(p):
    o, n = posixpath.split(p)
    return None if o + " old" in alle_ordner else f"{o} old/{n}"


def t4(p):
    o, n = posixpath.split(p)
    ziel = ordner_liste[rng_syn.integers(len(ordner_liste))]
    return None if ziel == o or f"{ziel}/{n}".lower() in alle_pfade_lower else f"{ziel}/{n}"


def t5(p):
    q = p.lower()
    return q if q != p else None


def t6(p):
    o, n = posixpath.split(p)
    try:
        m = n.encode("utf-8").decode("cp1252")
    except UnicodeDecodeError:
        return None
    return f"{o}/{m}" if m != n else None


def t7(p):
    o, n = posixpath.split(p)
    zahlen = list(re.finditer(r"\d+", n))
    if not zahlen:
        return None
    z = zahlen[-1]
    for d in range(1, 10):
        neu = n[:z.start()] + str(int(z.group()) + d).zfill(len(z.group())) + n[z.end():]
        if neu.lower() not in alle_namen:
            return f"{o}/{neu}"
    return None


def t8(p):
    n = posixpath.basename(p)
    m = re.search(r"(19|20)\d\d", n)
    if not m:
        return None
    neu = p.replace(m.group(), str(int(m.group()) + 1))
    return None if posixpath.basename(neu).lower() in alle_namen else neu


TYPEN = {"T1": (t1, False, "datei"), "T2": (t2, False, "datei"), "T3": (t3, True, "datei"),
         "T4": (t4, True, "datei"), "T5": (t5, False, "datei"), "T6": (t6, False, "datei"),
         "T7": (t7, False, "fehlt"), "T8": (t8, False, "fehlt")}

faelle = []
for i, (typ, (fkt, nur_eindeutig, erw)) in enumerate(TYPEN.items()):
    quelle = pool[pool["name_eindeutig"]] if nur_eindeutig else pool
    je_ordner, n = {}, 0
    for _, z in quelle.sample(frac=1, random_state=SEED + i).iterrows():
        if n == N_JE_TYP:
            break
        if je_ordner.get(z["ordner"], 0) >= MAX_JE_ORDNER:
            continue
        eintrag = fkt(z["pfad"])
        if eintrag is None:
            continue
        faelle.append({"typ": typ, "eintrag": eintrag, "original": z["pfad"],
                       "erwartung": z["pfad"] if erw == "datei" else "fehlt", "ordner": z["ordner"]})
        je_ordner[z["ordner"]] = je_ordner.get(z["ordner"], 0) + 1
        n += 1

df_syn = pd.DataFrame(faelle)
df_syn.to_csv(INTERIM / "03_06_synthetisch.csv", index=False, encoding="utf-8")   # privat
df_syn_zaehlung = (df_syn.groupby("typ", as_index=False)
                   .agg(faelle=("eintrag", "size"), ordner=("ordner", "nunique"),
                        erwartung_fehlt=("erwartung", lambda s: (s == "fehlt").sum())))
print(df_syn_zaehlung.to_string(index=False))
print(f"\nGesamt: {len(df_syn)} Fälle (Ziel {N_JE_TYP * len(TYPEN)})")
_ = tabelle_speichern(df_syn_zaehlung, "03_06_synthetisch_zaehlung")

## Notebook 03 – Abschluss: Gold-Set, Split, synthetische Testfälle (Phase 2)

**Datenbasis:** gespeicherte Zwischenstände aus Notebook 02 (8 Eingaben, Zeilenzahlen geprüft).

| Thema | Ergebnis | Entscheidung |
|---|---|---|
| Gold-Pilot (Zellen 02–04) | 20 Fälle „Kandidat prüfen" (K3 7, Q2 7, Q3 6), 20/20 richtig; Zeit je Label Median 21 s (KI 18–33 s) | kein umfangreiches Labeln jetzt; Precision „sicher" in Phase 3 aus der Ausgabe der Baseline, Blöcke zu 50, Ziel 300 (< 1 %), ca. 1,75 h |
| Split (Zelle 05) | 1.887 Fehltreffer, Fold = SHA-1(Ordner) mod 5; Testanteil Q2/Q3/Q5 22–23 %, K3 44,6 %, K4 11,4 %, Q4 2,9 % | Hash-Split für Q2/Q3/Q5; Leave-One-Group-Out für K3/K4/Q4 (Phase 4) |
| Synthetische Testfälle (Zelle 06) | 2.252 Fälle T1–T8 (T1 240, T6 212, sonst je 300), davon 600 harte Negativfälle (T7/T8) | Recall-Tests T1–T6, Precision-Tests T7/T8; Dokumentation `docs/generation_summary.md` |

**Grenzen:** Pilot nach Schicht sortiert (Lerneffekt und Schicht nicht trennbar); ein einzelner Bewerter; synthetische Fälle prüfen nur bekannte Fehlertypen.

**Phase 2 abgeschlossen.** Nächste Phase: 3 – deterministischer Kern (Baseline A).